# **EV Charging Station Optimization in Kenya**

In [150]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [151]:
#Loading the data
global_stations = pd.read_csv('../Data/charging_station.csv')
global_stations.head()

,id,name,city,state_province,country_code,latitude,longitude,ports,power_kw,power_class,is_fast_dc
0,307660,Av. de Tarragona,Andorra,UNKNOWN,AD,42.505254,1.528861,10,300.0,DC_ULTRA_(>=150kW),True
1,301207,Parquing Costa Rodona,Encamp,UNKNOWN,AD,42.537213,1.727014,10,22.0,AC_HIGH_(22-49kW),False
2,301206,Hotel Naudi,Unknown City,UNKNOWN,AD,42.576811,1.666061,1,11.0,AC_L2_(7.5-21kW),False
3,301205,Hotel Piolets Soldeu Centre,Unknown City,UNKNOWN,AD,42.576466,1.667317,1,22.0,AC_HIGH_(22-49kW),False
4,301204,Hotel Serras,Unknown City,UNKNOWN,AD,42.579458,1.659215,3,11.0,AC_L2_(7.5-21kW),False


In [152]:
country_summary_df= pd.read_csv('../Data/country_summary.csv')
country_summary_df.head()

,country_code,country,station_count,port_count,fast_station_share,fast_port_share
0,AD,Andorra,96,259,0.062500,0.138996
1,AE,United Arab Emirates,131,346,0.175573,0.410405
2,AF,Afghanistan,1,1,0.000000,0.000000
3,AL,Albania,15,16,0.600000,0.562500
4,AM,Armenia,4,6,0.250000,0.166667


In [153]:
world_summary_df= pd.read_csv('../Data/world_summary.csv')
world_summary_df.head()

,country_code,country,station_count,port_count,fast_station_count,fast_port_count,fast_station_share,fast_port_share,max_power_kw,median_power_kw,dc_fast_station_count,dc_ultra_station_count
0,AD,Andorra,96,259,6,36,0.062500,0.138996,300.0,22.0,6,2
1,AE,United Arab Emirates,131,346,23,142,0.175573,0.410405,250.0,22.0,23,16
2,AF,Afghanistan,1,1,0,0,0.000000,0.000000,22.0,22.0,0,0
3,AL,Albania,15,16,9,9,0.600000,0.562500,180.0,120.0,9,1
4,AM,Armenia,4,6,1,1,0.250000,0.166667,150.0,86.0,1,1


In [154]:
kenya_stations_sample = pd.read_csv('../Data/kenya_ev_charging_stations_sample_new.csv')
kenya_stations_sample.head()

,County/City,Station Name,Charger Type,Connector Types,Latitude,Longitude
0,Nairobi,Sarit Centre,AC + DC,"Type2, CHAdeMO",-1.261245,36.802243
1,Nairobi,Two Rivers Mall,AC,Type2,-1.210229,36.795216
2,Nairobi,Village Market Gigiri,AC,Type2,-1.227277,36.805228
3,Nairobi,UNGA House Westlands,AC,Type2,-1.262890,36.804670
4,Nairobi,KCB Towers Upper Hill,AC,Type2,-1.301320,36.812790


In [155]:
population = pd.read_csv('../Data/kenya-population-distibution-2019-census.csv')
population.head()

,County,Total,Male,Female,Intersex
0,KENYA,"14,831,700","7,352,134","7,478,883",683
1,Mombasa,"1,208,333","610,257","598,046",30
2,Kwale,"126,431","62,395","64,031",5
3,Kilifi,"393,888","191,324","202,558",6
4,Tana River,"75,722","37,854","37,867",1


In [156]:
admin_boundaries = pd.read_excel('../Data/ken_adminboundaries_tabulardata.xlsx')
admin_boundaries.head()

,OBJECTID *,Shape *,admin2Name_en,admin2Pcode,admin2RefName,admin2AltName1_en,admin2AltName2_en,admin1Name_en,admin1Pcode,admin0Name_en,admin0Pcode,date,validOn,validTo,Shape_Length,Shape_Area
0,1,Polygon,Ainabkoi,KE027144,<Null>,<Null>,<Null>,Uasin Gishu,KE027,Kenya,KE,2017-11-03,2018-06-07,<Null>,1.746986,0.040829
1,2,Polygon,Ainamoi,KE035190,<Null>,<Null>,<Null>,Kericho,KE035,Kenya,KE,2017-11-03,2018-06-07,<Null>,0.917307,0.019957
2,3,Polygon,Aldai,KE029152,<Null>,<Null>,<Null>,Nandi,KE029,Kenya,KE,2017-11-03,2018-06-07,<Null>,1.402637,0.038000
3,4,Polygon,Alego Usonga,KE041234,<Null>,<Null>,<Null>,Siaya,KE041,Kenya,KE,2017-11-03,2018-06-07,<Null>,1.081354,0.049357
4,5,Polygon,Awendo,KE044254,<Null>,<Null>,<Null>,Migori,KE044,Kenya,KE,2017-11-03,2018-06-07,<Null>,0.743915,0.021365


In [157]:
#Filtering the data to Kenya data only
kenya_global = global_stations[global_stations["country_code"] == "KE"].copy()
kenya_global.head()

,id,name,city,state_province,country_code,latitude,longitude,ports,power_kw,power_class,is_fast_dc
128395,267507,City Mall EvChaja,Unknown City,Mombasa,KE,-4.020064,39.720491,1,22.0,AC_HIGH_(22-49kW),False
128396,267505,Waiyaki Way,Unknown City,Nairobi,KE,-1.259614,36.776883,1,22.0,AC_HIGH_(22-49kW),False
128397,194150,Waterfront Mall EvChaja,Unknown City,Nairobi,KE,-1.329425,36.715820,1,22.0,AC_HIGH_(22-49kW),False
128398,194149,Two Rivers Mall EvChaja,Gachie,Nairobi,KE,-1.210229,36.795216,1,11.0,AC_L2_(7.5-21kW),False
128399,189881,Nopea Ride Village Market,Nairobi,Nairobi,KE,-1.227277,36.805228,1,22.0,AC_HIGH_(22-49kW),False


In [158]:
county_coords = pd.read_csv('../Data/county_coordinates.csv')
county_coords.head()

,county,Latitude,Longitude
0,Uasin Gishu,0.516667,35.283300
1,Kericho,-0.367740,35.283140
2,Nandi,0.166667,35.150000
3,Siaya,-0.083333,34.249999
4,Nakuru,-0.283332,36.066666


In [159]:
kenya_stations_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   County/City      16 non-null     object 
 1   Station Name     16 non-null     object 
 2   Charger Type     16 non-null     object 
 3   Connector Types  16 non-null     object 
 4   Latitude         16 non-null     float64
 5   Longitude        16 non-null     float64
dtypes: float64(2), object(4)
memory usage: 896.0+ bytes


In [160]:
population.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   County    48 non-null     object
 1   Total     48 non-null     object
 2   Male      48 non-null     object
 3   Female    48 non-null     object
 4   Intersex  48 non-null     object
dtypes: object(5)
memory usage: 2.0+ KB


In [161]:
admin_boundaries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   OBJECTID *         290 non-null    int64         
 1   Shape *            290 non-null    object        
 2   admin2Name_en      290 non-null    object        
 3   admin2Pcode        290 non-null    object        
 4   admin2RefName      290 non-null    object        
 5   admin2AltName1_en  290 non-null    object        
 6   admin2AltName2_en  290 non-null    object        
 7   admin1Name_en      290 non-null    object        
 8   admin1Pcode        290 non-null    object        
 9   admin0Name_en      290 non-null    object        
 10  admin0Pcode        290 non-null    object        
 11  date               290 non-null    datetime64[ns]
 12  validOn            290 non-null    datetime64[ns]
 13  validTo            290 non-null    object        
 14  Shape_Leng

In [162]:
county_coords.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   county     47 non-null     object 
 1   Latitude   47 non-null     float64
 2   Longitude  47 non-null     float64
dtypes: float64(2), object(1)
memory usage: 1.2+ KB


## Cleaning

In [163]:
kenya_global.duplicated().sum()

0

In [164]:
kenya_global.isnull().sum()

id                0
name              0
city              0
state_province    0
country_code      0
latitude          0
longitude         0
ports             0
power_kw          0
power_class       0
is_fast_dc        0
dtype: int64

In [165]:
kenya_stations_sample.duplicated().sum()

0

In [166]:
kenya_stations_sample.isnull().sum()

County/City        0
Station Name       0
Charger Type       0
Connector Types    0
Latitude           0
Longitude          0
dtype: int64

In [167]:
population.duplicated().sum()

0

In [168]:
population.isnull().sum()

County      0
Total       0
Male        0
Female      0
Intersex    0
dtype: int64

In [169]:
admin_boundaries.duplicated().sum()

0

In [170]:
admin_boundaries.isnull().sum()

OBJECTID *           0
Shape *              0
admin2Name_en        0
admin2Pcode          0
admin2RefName        0
admin2AltName1_en    0
admin2AltName2_en    0
admin1Name_en        0
admin1Pcode          0
admin0Name_en        0
admin0Pcode          0
date                 0
validOn              0
validTo              0
Shape_Length         0
Shape_Area           0
dtype: int64

In [171]:
county_coords.duplicated().sum()

0

In [172]:
county_coords.isnull().sum()

county       0
Latitude     0
Longitude    0
dtype: int64

## Preparation

In [173]:
kenya_global.head()

,id,name,city,state_province,country_code,latitude,longitude,ports,power_kw,power_class,is_fast_dc
128395,267507,City Mall EvChaja,Unknown City,Mombasa,KE,-4.020064,39.720491,1,22.0,AC_HIGH_(22-49kW),False
128396,267505,Waiyaki Way,Unknown City,Nairobi,KE,-1.259614,36.776883,1,22.0,AC_HIGH_(22-49kW),False
128397,194150,Waterfront Mall EvChaja,Unknown City,Nairobi,KE,-1.329425,36.715820,1,22.0,AC_HIGH_(22-49kW),False
128398,194149,Two Rivers Mall EvChaja,Gachie,Nairobi,KE,-1.210229,36.795216,1,11.0,AC_L2_(7.5-21kW),False
128399,189881,Nopea Ride Village Market,Nairobi,Nairobi,KE,-1.227277,36.805228,1,22.0,AC_HIGH_(22-49kW),False


In [174]:
kenya_global = kenya_global.drop(columns=['id', 'city', 'is_fast_dc'])
kenya_global.head()

,name,state_province,country_code,latitude,longitude,ports,power_kw,power_class
128395,City Mall EvChaja,Mombasa,KE,-4.020064,39.720491,1,22.0,AC_HIGH_(22-49kW)
128396,Waiyaki Way,Nairobi,KE,-1.259614,36.776883,1,22.0,AC_HIGH_(22-49kW)
128397,Waterfront Mall EvChaja,Nairobi,KE,-1.329425,36.715820,1,22.0,AC_HIGH_(22-49kW)
128398,Two Rivers Mall EvChaja,Nairobi,KE,-1.210229,36.795216,1,11.0,AC_L2_(7.5-21kW)
128399,Nopea Ride Village Market,Nairobi,KE,-1.227277,36.805228,1,22.0,AC_HIGH_(22-49kW)


In [175]:
kenya_global = kenya_global.rename(columns={"state_province": "county", "name": "station name"})

In [176]:
kenya_global['county'].value_counts()

Nairobi    11
Mombasa     1
Name: county, dtype: int64

In [177]:
kenya_global["charger type"] = kenya_global["power_class"].str.split("_").str[0].str.upper()

In [178]:
kenya_global = kenya_global.drop(columns = ["power_class", "power_kw"])

In [179]:
kenya_global.head(2)

,station name,county,country_code,latitude,longitude,ports,charger type
128395,City Mall EvChaja,Mombasa,KE,-4.020064,39.720491,1,AC
128396,Waiyaki Way,Nairobi,KE,-1.259614,36.776883,1,AC


In [180]:
kenya_stations_sample.head(2)

,County/City,Station Name,Charger Type,Connector Types,Latitude,Longitude
0,Nairobi,Sarit Centre,AC + DC,"Type2, CHAdeMO",-1.261245,36.802243
1,Nairobi,Two Rivers Mall,AC,Type2,-1.210229,36.795216


In [181]:
kenya_stations_sample =kenya_stations_sample.drop(columns='Connector Types')

In [182]:
kenya_stations_sample = kenya_stations_sample.rename(columns={"County/City": "county", "Station Name": "station name", "Charger Type": "charger type", "Latitude": "latitude", "Longitude": "longitude"})

In [183]:
kenya_stations_sample['county'].value_counts()

Nairobi     9
Mombasa     1
Kisumu      1
Kiambu      1
Narok       1
Mavoko      1
Naivasha    1
Nakuru      1
Name: county, dtype: int64

In [184]:
kenya_stations_sample['county'] = kenya_stations_sample['county'].replace(['Mavoko', 'Naivasha'], ['Machakos', 'Nakuru'])

In [185]:
kenya_stations_sample['county'].value_counts()

Nairobi     9
Nakuru      2
Machakos    1
Mombasa     1
Kisumu      1
Kiambu      1
Narok       1
Name: county, dtype: int64

In [186]:
kenya_stations_sample.head(2)

,county,station name,charger type,latitude,longitude
0,Nairobi,Sarit Centre,AC + DC,-1.261245,36.802243
1,Nairobi,Two Rivers Mall,AC,-1.210229,36.795216


In [187]:
population

,County,Total,Male,Female,Intersex
0,KENYA,"14,831,700","7,352,134","7,478,883",683
1,Mombasa,"1,208,333","610,257","598,046",30
2,Kwale,"126,431","62,395","64,031",5
3,Kilifi,"393,888","191,324","202,558",6
4,Tana River,"75,722","37,854","37,867",1
5,Lamu,"38,446","19,533","18,911",2
6,Taita/Taveta,"93,774","46,620","47,149",5
7,Garissa,"210,890","109,552","101,331",7
8,Wajir,"177,174","94,812","82,340",22
9,Mandera,"270,467","135,548","134,909",10


In [188]:
population = population.drop(columns=['Male', 'Female', 'Intersex'])

In [189]:
population = population.rename(columns={"County": "county", "Total": "population"})

In [190]:
population = population[population['county'] != 'KENYA']

In [191]:
population['county'] = population['county'].replace(['Nairobi City', 'Elg eyo/Marakwet', 'Taita/Taveta'], ['Nairobi', 'Elgeyo-Marakwet', 'Taita Taveta'])

In [192]:
population['population'] = population['population'].str.replace(',', '').astype(int)

In [193]:
population.head(2)

,county,population
1,Mombasa,1208333
2,Kwale,126431


In [194]:
admin_boundaries.head(2)

,OBJECTID *,Shape *,admin2Name_en,admin2Pcode,admin2RefName,admin2AltName1_en,admin2AltName2_en,admin1Name_en,admin1Pcode,admin0Name_en,admin0Pcode,date,validOn,validTo,Shape_Length,Shape_Area
0,1,Polygon,Ainabkoi,KE027144,<Null>,<Null>,<Null>,Uasin Gishu,KE027,Kenya,KE,2017-11-03,2018-06-07,<Null>,1.746986,0.040829
1,2,Polygon,Ainamoi,KE035190,<Null>,<Null>,<Null>,Kericho,KE035,Kenya,KE,2017-11-03,2018-06-07,<Null>,0.917307,0.019957


In [195]:
admin_boundaries = admin_boundaries.rename(columns={"admin1Name_en": "county","admin2Name_en": "sub county", "admin0Pcode": "country code", "Shape_Area": "shape area", "Shape_Length": "shape length"})

In [196]:
admin_boundaries = admin_boundaries[['county', 'sub county', 'country code', 'shape area', 'shape length']]

In [197]:
admin_boundaries

,county,sub county,country code,shape area,shape length
0,Uasin Gishu,Ainabkoi,KE,0.040829,1.746986
1,Kericho,Ainamoi,KE,0.019957,0.917307
2,Nandi,Aldai,KE,0.038000,1.402637
3,Siaya,Alego Usonga,KE,0.049357,1.081354
4,Migori,Awendo,KE,0.021365,0.743915
...,...,...,...,...,...
285,Bungoma,Webuye West,KE,0.019180,1.010876
286,Nyamira,West Mugirango,KE,0.014610,0.630183
287,Nairobi,Westlands,KE,0.005908,0.405298
288,Taita Taveta,Wundanyi,KE,0.069243,1.214526


In [198]:
county_coords.head(2)

,county,Latitude,Longitude
0,Uasin Gishu,0.516667,35.28330
1,Kericho,-0.367740,35.28314


In [199]:
county_coords = county_coords.rename(columns={"Latitude": "county_lat", "Longitude": "county_lon"})

In [200]:
county_coords.head(2)

,county,county_lat,county_lon
0,Uasin Gishu,0.516667,35.28330
1,Kericho,-0.367740,35.28314


In [201]:
#merging all Kenyan stations. combine rows from kenya_global and kenya_stations_sample
kenya_stations = pd.concat([kenya_global, kenya_stations_sample], ignore_index=True)
kenya_stations

,station name,county,country_code,latitude,longitude,ports,charger type
0,City Mall EvChaja,Mombasa,KE,-4.020064,39.720491,1.0,AC
1,Waiyaki Way,Nairobi,KE,-1.259614,36.776883,1.0,AC
2,Waterfront Mall EvChaja,Nairobi,KE,-1.329425,36.715820,1.0,AC
3,Two Rivers Mall EvChaja,Nairobi,KE,-1.210229,36.795216,1.0,AC
4,Nopea Ride Village Market,Nairobi,KE,-1.227277,36.805228,1.0,AC
5,Nopea JKIA Airport,Nairobi,KE,-1.333731,36.928334,1.0,AC
6,Two Rivers Mall Nairobi (Nopea),Nairobi,KE,-1.211449,36.795808,1.0,AC
7,Sarit Centre Nairobi,Nairobi,KE,-1.261245,36.802243,1.0,AC
8,TRM Mall,Nairobi,KE,-1.219824,36.889224,2.0,AC
9,Holy Family Basilica Parking Silo,Nairobi,KE,-1.287358,36.820061,1.0,AC


In [202]:
#Omit stations that appear twice
kenya_stations = kenya_stations.drop(index=[12, 13, 14, 25])
kenya_stations

,station name,county,country_code,latitude,longitude,ports,charger type
0,City Mall EvChaja,Mombasa,KE,-4.020064,39.720491,1.0,AC
1,Waiyaki Way,Nairobi,KE,-1.259614,36.776883,1.0,AC
2,Waterfront Mall EvChaja,Nairobi,KE,-1.329425,36.715820,1.0,AC
3,Two Rivers Mall EvChaja,Nairobi,KE,-1.210229,36.795216,1.0,AC
4,Nopea Ride Village Market,Nairobi,KE,-1.227277,36.805228,1.0,AC
5,Nopea JKIA Airport,Nairobi,KE,-1.333731,36.928334,1.0,AC
6,Two Rivers Mall Nairobi (Nopea),Nairobi,KE,-1.211449,36.795808,1.0,AC
7,Sarit Centre Nairobi,Nairobi,KE,-1.261245,36.802243,1.0,AC
8,TRM Mall,Nairobi,KE,-1.219824,36.889224,2.0,AC
9,Holy Family Basilica Parking Silo,Nairobi,KE,-1.287358,36.820061,1.0,AC


In [203]:
kenya_stations.isnull().sum()

station name     0
county           0
country_code    12
latitude         0
longitude        0
ports           12
charger type     0
dtype: int64

In [204]:
kenya_stations['country_code'] = kenya_stations['country_code'].fillna('KE')

In [205]:
kenya_stations['ports'] = kenya_stations['ports'].fillna(1.0)

In [206]:
kenya_stations.isnull().sum()

station name    0
county          0
country_code    0
latitude        0
longitude       0
ports           0
charger type    0
dtype: int64

In [207]:
kenya_stations

,station name,county,country_code,latitude,longitude,ports,charger type
0,City Mall EvChaja,Mombasa,KE,-4.020064,39.720491,1.0,AC
1,Waiyaki Way,Nairobi,KE,-1.259614,36.776883,1.0,AC
2,Waterfront Mall EvChaja,Nairobi,KE,-1.329425,36.715820,1.0,AC
3,Two Rivers Mall EvChaja,Nairobi,KE,-1.210229,36.795216,1.0,AC
4,Nopea Ride Village Market,Nairobi,KE,-1.227277,36.805228,1.0,AC
5,Nopea JKIA Airport,Nairobi,KE,-1.333731,36.928334,1.0,AC
6,Two Rivers Mall Nairobi (Nopea),Nairobi,KE,-1.211449,36.795808,1.0,AC
7,Sarit Centre Nairobi,Nairobi,KE,-1.261245,36.802243,1.0,AC
8,TRM Mall,Nairobi,KE,-1.219824,36.889224,2.0,AC
9,Holy Family Basilica Parking Silo,Nairobi,KE,-1.287358,36.820061,1.0,AC


In [208]:
station_counts = kenya_stations.groupby('county').size().reset_index(name='num_stations')
station_counts

,county,num_stations
0,Kiambu,1
1,Kisumu,1
2,Machakos,1
3,Mombasa,1
4,Nairobi,17
5,Nakuru,2
6,Narok,1


In [209]:
#merging population and stations data
county_df = county_coords.merge(population, on='county', how='left')
county_df.head()

,county,county_lat,county_lon,population
0,Uasin Gishu,0.516667,35.283300,510205
1,Kericho,-0.367740,35.283140,93538
2,Nandi,0.166667,35.150000,59479
3,Siaya,-0.083333,34.249999,85417
4,Nakuru,-0.283332,36.066666,1047080


In [210]:
county_df = county_df.merge(station_counts, on='county', how='left')
county_df.head()

,county,county_lat,county_lon,population,num_stations
0,Uasin Gishu,0.516667,35.283300,510205,NaN
1,Kericho,-0.367740,35.283140,93538,NaN
2,Nandi,0.166667,35.150000,59479,NaN
3,Siaya,-0.083333,34.249999,85417,NaN
4,Nakuru,-0.283332,36.066666,1047080,2.0


In [211]:
county_df['num_stations'] = county_df['num_stations'].fillna(0)

In [212]:
county_df.head(15)

,county,county_lat,county_lon,population,num_stations
0,Uasin Gishu,0.516667,35.283300,510205,0.0
1,Kericho,-0.367740,35.283140,93538,0.0
2,Nandi,0.166667,35.150000,59479,0.0
3,Siaya,-0.083333,34.249999,85417,0.0
4,Nakuru,-0.283332,36.066666,1047080,2.0
5,Garissa,-0.452750,39.646010,210890,0.0
6,Mandera,3.416670,40.666700,270467,0.0
7,Baringo,0.466670,35.966670,75289,0.0
8,Kisii,-0.681740,34.766660,151410,0.0
9,Bomet,-0.800000,35.233300,27971,0.0
